In [1]:
import pandas as pd
import matplotlib as plt
import numpy as np
from datetime import datetime

In [2]:
TARGET_COLUMNS = [
'Agronomist Name',
'Farmer',
'Country',
'Region',	
'Province',	
'Status',	
'Engaged Area',
'Agronomist Join Date',
'Farmer Creation Date',
'Acquisition Channel',
'Cohort',
'link'

]


In [ ]:
df = pd.read_excel(
    r'file_path\list_of_all_farmers_2026-04-02T10_26_21.662688Z.xlsx',
    sheet_name="Query result"
)

In [4]:
df.columns = df.columns.str.strip()

In [5]:
df = df[[col for col in TARGET_COLUMNS if col in df.columns]]

In [6]:
df.columns = df.columns.str.replace(' ', '_')

In [7]:
df.columns

Index(['Agronomist_Name', 'Farmer', 'Country', 'Region', 'Province', 'Status',
       'Engaged_Area', 'Agronomist_Join_Date', 'Farmer_Creation_Date',
       'Acquisition_Channel', 'Cohort', 'link'],
      dtype='str')

In [ ]:
Channel_1_AGRONOMISTS = [
    'agronomist_1',
    'agronomist_2',
    'agronomist_3',
    'agronomist_4',
    'agronomist_5',
]


In [ ]:
def assign_channel(Agronomist_Name):
    if pd.isna(Agronomist_Name):
        return "Channel_0"
    elif Agronomist_Name in Channel_1_AGRONOMISTS:
        return "Channel_1"
    elif Agronomist_Name == "C_prod":
        return "Channel_2"
    else:
        return "Channel_0"

In [10]:
df.insert(
    loc=df.columns.get_loc('Farmer'),
    column='Channel',
    value=df['Agronomist_Name'].apply(assign_channel)
)

In [11]:
df.columns

Index(['Agronomist_Name', 'Channel', 'Farmer', 'Country', 'Region', 'Province',
       'Status', 'Engaged_Area', 'Agronomist_Join_Date',
       'Farmer_Creation_Date', 'Acquisition_Channel', 'Cohort', 'link'],
      dtype='str')

In [12]:
def Engaged_categories(Engaged_Area):
    if Engaged_Area == 0:
        return 'No Parcel'
    elif Engaged_Area < 0.6:
        return '<0.6 Ha'
    elif Engaged_Area <= 1:
        return'0.6 --> 1 HA'
    elif Engaged_Area <= 5:
        return '1 --> 5 Ha'
    elif Engaged_Area <= 15:
        return '5 --> 15 Ha'
    else:
        return 'Above 15 Ha'

In [13]:
df.insert(
    loc=df.columns.get_loc('Agronomist_Join_Date'),
    column='total_engaged_categories',
    value=df['Engaged_Area'].apply(Engaged_categories)

)

In [14]:
def Farmer_Eligibility(Engaged_Area):
    if Engaged_Area == 0:
        return 'No Parcel'
    elif Engaged_Area < 0.6:
        return 'Below 0.6 HA'
    else:
        return 'Above 0.6 HA'

In [15]:
df.insert(
    loc=df.columns.get_loc('total_engaged_categories'),
    column='Farmer_Eligibility',
    value=df['Engaged_Area'].apply(Farmer_Eligibility)
)

In [ ]:
"""extract unique farmers_id from the link, this is will be a primary key here
and a foreing key in the parcels_data
"""
df.insert(
    loc=df.columns.get_loc('link'),
    column='Farmer_ID',
    value=df['link'].str.extract(r'farmers/(.*?)/profile')
)

In [18]:
df.columns

Index(['Agronomist_Name', 'Channel', 'Farmer', 'Country', 'Region', 'Province',
       'Status', 'Engaged_Area', 'Farmer_Eligibility',
       'total_engaged_categories', 'Agronomist_Join_Date',
       'Farmer_Creation_Date', 'Acquisition_Channel', 'Cohort', 'Farmer_ID',
       'link'],
      dtype='str')

In [ ]:
#export to to file for further analysis and visualisation on excel
todays=datetime.today().strftime('%d_%m_%Y')
df.to_excel(f"All_Farmers_cleaned_{todays}.xlsx", index=False)